# AnanthiX AI - Jalon 2 : Preprocessing + Training Baseline

**Objectif** : Préparer les données et entraîner ResNet-50 baseline

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from pathlib import Path
from collections import Counter
import pickle
import tensorflow_datasets as tfds
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

BASE_PATH = Path('/content/drive/MyDrive/AnanthiX_AI')
DATA_PATH = BASE_PATH / 'data'
RESULTS_PATH = BASE_PATH / 'results'
MODELS_PATH = BASE_PATH / 'models'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ÉTAPE 1 : Charger metadata

print('Chargement metadata...')

metadata_path = DATA_PATH / 'plantvillage_metadata.json'
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

class_names = metadata['class_names']
num_classes = len(class_names)

print(f'✓ Classes: {num_classes}')

# ÉTAPE 2 : Charger TOUT le dataset en mémoire

print('\nChargement du dataset PlantVillage en mémoire...')

ds, info = tfds.load('plant_village', with_info=True, as_supervised=True)
train_ds = ds['train']

all_images = []
all_labels = []

print('Lecture du dataset...')
for i, (image, label) in enumerate(train_ds):
    all_images.append(image.numpy().astype(np.uint8))
    all_labels.append(label.numpy())

    if (i + 1) % 10000 == 0:
        print(f'  {i + 1}/{info.splits["train"].num_examples}')

all_images = np.array(all_images)
all_labels = np.array(all_labels)

print(f'✓ Dataset en RAM: {all_images.shape}')

# ÉTAPE 3 : Créer splits stratifiés

print('\nCréation splits train/val/test...')

train_indices = []
val_indices = []
test_indices = []

for class_idx in range(num_classes):
    class_mask = all_labels == class_idx
    class_indices = np.where(class_mask)[0]

    n = len(class_indices)
    n_train = int(0.7 * n)
    n_val = int(0.15 * n)

    train_indices.extend(class_indices[:n_train])
    val_indices.extend(class_indices[n_train:n_train + n_val])
    test_indices.extend(class_indices[n_train + n_val:])

print(f'✓ Train: {len(train_indices)} | Val: {len(val_indices)} | Test: {len(test_indices)}')

# ÉTAPE 4 : PyTorch Dataset & DataLoaders

print('\nCréation DataLoaders...')

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, scale=(0.8, 1.2)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(256, pad_if_needed=True),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

val_test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

class FastDataset(Dataset):
    def __init__(self, images, labels, indices, transform=None):
        self.images = images
        self.labels = labels
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        actual_idx = self.indices[idx]
        image = self.images[actual_idx]
        label = self.labels[actual_idx]

        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)

        return image, label

# Batch size optimisé pour G4 (16GB VRAM)
batch_size = 48

train_dataset = FastDataset(all_images, all_labels, train_indices, transform=train_transform)
val_dataset = FastDataset(all_images, all_labels, val_indices, transform=val_test_transform)
test_dataset = FastDataset(all_images, all_labels, test_indices, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f'✓ DataLoaders: train {len(train_loader)} | val {len(val_loader)} | test {len(test_loader)}')

# ÉTAPE 5 : Charger modèle

print('\nChargement ResNet-50...')

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(2048, num_classes)

for param in model.layer1.parameters():
    param.requires_grad = False
for param in model.layer2.parameters():
    param.requires_grad = False

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✓ Modèle chargé | Trainable params: {trainable:,}')

# ÉTAPE 6 : Configuration training (FIX: sans verbose)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

print('✓ Configuration: CrossEntropyLoss + Adam + Scheduler')

# ÉTAPE 7 : BOUCLE TRAINING OPTIMISÉE POUR G4

print('\nDémarrage entraînement (G4 GPU)...\n')

num_epochs = 10
best_val_loss = float('inf')
patience_counter = 0

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    # TRAIN
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss /= len(train_loader)
    train_acc = 100 * train_correct / train_total

    # VAL
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = 100 * val_correct / val_total

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1

    print(f'Epoch {epoch+1:2d}/10 | TL: {train_loss:.4f} ({train_acc:5.1f}%) | VL: {val_loss:.4f} ({val_acc:5.1f}%)')

    if patience_counter >= 3:
        print(f'Early stopping at epoch {epoch+1}')
        break

print('\n✓ Training complété')

# ÉTAPE 8 : Évaluation TEST

print('\nÉvaluation test set...')

model.eval()
test_correct, test_total = 0, 0
all_preds, all_labels_test = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels_test.extend(labels.cpu().numpy())

test_acc = 100 * test_correct / test_total
f1_macro = f1_score(all_labels_test, all_preds, average='macro', zero_division=0)
f1_weighted = f1_score(all_labels_test, all_preds, average='weighted', zero_division=0)
precision = precision_score(all_labels_test, all_preds, average='macro', zero_division=0)
recall = recall_score(all_labels_test, all_preds, average='macro', zero_division=0)

cm = confusion_matrix(all_labels_test, all_preds)

print(f'\n✓ TEST RESULTS:')
print(f'  - Accuracy: {test_acc:.2f}%')
print(f'  - F1 Macro: {f1_macro:.4f}')
print(f'  - F1 Weighted: {f1_weighted:.4f}')
print(f'  - Precision: {precision:.4f}')
print(f'  - Recall: {recall:.4f}')

# ÉTAPE 9 : Sauvegarder résultats

print('\nSauvegarde dans Google Drive...')

torch.save(model.state_dict(), MODELS_PATH / 'resnet50_baseline.pth')

with open(RESULTS_PATH / 'training_history.json', 'w') as f:
    json.dump(history, f)

metrics = {
    'test_accuracy': float(test_acc),
    'f1_macro': float(f1_macro),
    'f1_weighted': float(f1_weighted),
    'precision': float(precision),
    'recall': float(recall),
    'epochs_trained': len(history['train_loss'])
}

with open(RESULTS_PATH / 'metrics_baseline.json', 'w') as f:
    json.dump(metrics, f, indent=2)

with open(RESULTS_PATH / 'confusion_matrix.pkl', 'wb') as f:
    pickle.dump(cm, f)

print(f'✓ Modèle: {MODELS_PATH / "resnet50_baseline.pth"}')
print(f'✓ Métriques: {RESULTS_PATH / "metrics_baseline.json"}')
print(f'✓ History: {RESULTS_PATH / "training_history.json"}')


# ÉTAPE 10 : Visualiser training curves

print('\nGénération visualisations...')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train', marker='o', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', marker='s', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train', marker='o', linewidth=2)
axes[1].plot(history['val_acc'], label='Val', marker='s', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Training Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curves_path = RESULTS_PATH / 'training_curves_baseline.png'
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.close()

print(f'✓ Curves: {curves_path}')
print('\n✓ ÉTAPE 2 COMPLÉTÉE AVEC SUCCÈS')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Chargement metadata...
✓ Classes: 38

Chargement du dataset PlantVillage en mémoire...
Lecture du dataset...
  10000/54303
  20000/54303
  30000/54303
  40000/54303
  50000/54303
✓ Dataset en RAM: (54303, 256, 256, 3)

Création splits train/val/test...
✓ Train: 37995 | Val: 8129 | Test: 8179

Création DataLoaders...
✓ DataLoaders: train 792 | val 170 | test 171

Chargement ResNet-50...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 252MB/s]

✓ Modèle chargé | Trainable params: 22,150,502
✓ Configuration: CrossEntropyLoss + Adam + Scheduler

Démarrage entraînement (G4 GPU)...



Epoch  1/10 | TL: 0.3659 ( 90.8%) | VL: 0.0374 ( 98.9%)
Epoch  2/10 | TL: 0.0425 ( 98.7%) | VL: 0.0202 ( 99.4%)
Epoch  3/10 | TL: 0.0296 ( 99.1%) | VL: 0.0184 ( 99.5%)
Epoch  4/10 | TL: 0.0222 ( 99.3%) | VL: 0.0213 ( 99.3%)
Epoch  5/10 | TL: 0.0162 ( 99.5%) | VL: 0.0141 ( 99.5%)
Epoch  6/10 | TL: 0.0167 ( 99.5%) | VL: 0.0169 ( 99.4%)
Epoch  7/10 | TL: 0.0117 ( 99.6%) | VL: 0.0117 ( 99.7%)
Epoch  8/10 | TL: 0.0125 ( 99.6%) | VL: 0.0102 ( 99.7%)
Epoch  9/10 | TL: 0.0122 ( 99.6%) | VL: 0.0123 ( 99.6%)
Epoch 10/10 | TL: 0.0096 ( 99.7%) | VL: 0.0177 ( 99.5%)

✓ Training complété

Évaluation test set...

✓ TEST RESULTS:
  - Accuracy: 99.43%
  - F1 Macro: 0.9908
  - F1 Weighted: 0.9942
  - Precision: 0.9912
  - Recall: 0.9908

Sauvegarde dans Google Drive...
✓ Modèle: /content/drive/MyDrive/AnanthiX_AI/models/resnet50_baseline.pth
✓ Métriques: /content/drive/MyDrive/AnanthiX_AI/results/metrics_baseline.json
✓ History: /content/drive/MyDrive/AnanthiX_AI/results/training_history.json

Génératio

In [ ]:
import json
from pathlib import Path

notebook_content = {
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# AnanthiX AI - Jalon 2 : Evaluation Complète\n",
                "\n",
                "**Objectif** : Analyser les erreurs du modèle baseline\n",
                "\n",
                "## Outputs\n",
                "- Confusion matrix (38×38)\n",
                "- Per-class metrics (P/R/F1)\n",
                "- Top 20 erreurs visualisées"
            ]
        }
    ] + [
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [code]
        }
        for code in [
            "import json\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torchvision.models as models\nfrom torch.utils.data import DataLoader, Dataset\nimport torchvision.transforms as transforms\nfrom pathlib import Path\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom sklearn.metrics import confusion_matrix\nimport pickle\nimport tensorflow_datasets as tfds\nfrom google.colab import drive\n\ndrive.mount('/content/drive', force_remount=False)\nBASE_PATH = Path('/content/drive/MyDrive/AnanthiX_AI')\nDATA_PATH = BASE_PATH / 'data'\nRESULTS_PATH = BASE_PATH / 'results'\nMODELS_PATH = BASE_PATH / 'models'\n\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nprint(f'Device: {device}')",
            "with open(DATA_PATH / 'plantvillage_metadata.json', 'r') as f:\n    metadata = json.load(f)\n\nclass_names = metadata['class_names']\nnum_classes = len(class_names)\n\nprint('Chargement dataset...')\nds, info = tfds.load('plant_village', with_info=True, as_supervised=True)\ntrain_ds = ds['train']\n\nall_images = []\nall_labels = []\n\nfor i, (image, label) in enumerate(train_ds):\n    all_images.append(image.numpy().astype(np.uint8))\n    all_labels.append(label.numpy())\n    if (i + 1) % 10000 == 0:\n        print(f'  {i + 1}/54303')\n\nall_images = np.array(all_images)\nall_labels = np.array(all_labels)\n\nprint(f'✓ Dataset chargé')",
            "print('Création splits...')\n\ntrain_indices = []\nval_indices = []\ntest_indices = []\n\nfor class_idx in range(num_classes):\n    mask = all_labels == class_idx\n    indices = np.where(mask)[0]\n    n = len(indices)\n    n_train = int(0.7 * n)\n    n_val = int(0.15 * n)\n    \n    train_indices.extend(indices[:n_train])\n    val_indices.extend(indices[n_train:n_train + n_val])\n    test_indices.extend(indices[n_train + n_val:])\n\nprint(f'✓ Splits créés')",
            "imagenet_mean = [0.485, 0.456, 0.406]\nimagenet_std = [0.229, 0.224, 0.225]\n\ntest_transform = transforms.Compose([\n    transforms.ToPILImage(),\n    transforms.ToTensor(),\n    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)\n])\n\nclass FastDataset(Dataset):\n    def __init__(self, images, labels, indices, transform=None):\n        self.images = images\n        self.labels = labels\n        self.indices = indices\n        self.transform = transform\n    \n    def __len__(self):\n        return len(self.indices)\n    \n    def __getitem__(self, idx):\n        actual_idx = self.indices[idx]\n        image = self.images[actual_idx]\n        label = self.labels[actual_idx]\n        \n        if self.transform:\n            image = self.transform(image)\n        else:\n            image = transforms.ToTensor()(image)\n        \n        return image, label\n\ntest_dataset = FastDataset(all_images, all_labels, test_indices, transform=test_transform)\ntest_loader = DataLoader(test_dataset, batch_size=48, shuffle=False, num_workers=2, pin_memory=True)\n\nprint(f'✓ Test loader: {len(test_loader)} batches')",
            "print('Chargement modèle...')\n\nmodel = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)\nmodel.fc = nn.Linear(2048, num_classes)\nmodel.load_state_dict(torch.load(MODELS_PATH / 'resnet50_baseline.pth'))\nmodel = model.to(device)\nmodel.eval()\n\nprint(f'✓ Modèle chargé')",
            "print('Évaluation...')\n\nall_preds = []\nall_labels_true = []\nall_confidence = []\nall_images_wrong = []\nall_wrong_preds = []\nall_wrong_truths = []\nall_wrong_conf = []\n\nwith torch.no_grad():\n    for images, labels in test_loader:\n        images = images.to(device)\n        outputs = model(images)\n        probs = torch.softmax(outputs, dim=1)\n        confidence, predicted = torch.max(probs, 1)\n        \n        all_preds.extend(predicted.cpu().numpy())\n        all_labels_true.extend(labels.numpy())\n        all_confidence.extend(confidence.cpu().numpy())\n        \n        wrong_mask = predicted.cpu().numpy() != labels.numpy()\n        if wrong_mask.any():\n            for i, is_wrong in enumerate(wrong_mask):\n                if is_wrong:\n                    all_images_wrong.append(images[i].cpu())\n                    all_wrong_preds.append(predicted[i].item())\n                    all_wrong_truths.append(labels[i].item())\n                    all_wrong_conf.append(confidence[i].item())\n\nall_preds = np.array(all_preds)\nall_labels_true = np.array(all_labels_true)\nall_confidence = np.array(all_confidence)\n\naccuracy = 100 * np.mean(all_preds == all_labels_true)\nprint(f'✓ Accuracy: {accuracy:.2f}%')",
            "print('Confusion matrix...')\n\ncm = confusion_matrix(all_labels_true, all_preds)\n\nfig, ax = plt.subplots(figsize=(16, 14))\nsns.heatmap(cm, annot=False, cmap='Blues', cbar=True, ax=ax, xticklabels=False, yticklabels=False)\nax.set_xlabel('Prédiction', fontsize=12)\nax.set_ylabel('Vérité', fontsize=12)\nax.set_title('Confusion Matrix - PlantVillage (38×38)', fontsize=14, fontweight='bold')\n\ncm_path = RESULTS_PATH / 'confusion_matrix_full.png'\nplt.savefig(cm_path, dpi=150, bbox_inches='tight')\nplt.close()\n\nprint(f'✓ Confusion matrix: {cm_path}')",
            "print('Per-class metrics...')\n\nper_class_metrics = {}\n\nfor class_idx in range(num_classes):\n    tp = cm[class_idx, class_idx]\n    fp = cm[:, class_idx].sum() - tp\n    fn = cm[class_idx, :].sum() - tp\n    \n    precision = tp / (tp + fp) if (tp + fp) > 0 else 0\n    recall = tp / (tp + fn) if (tp + fn) > 0 else 0\n    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0\n    \n    per_class_metrics[class_names[class_idx]] = {\n        'precision': float(precision),\n        'recall': float(recall),\n        'f1': float(f1),\n        'support': int(cm[class_idx].sum())\n    }\n\nwith open(RESULTS_PATH / 'per_class_metrics.json', 'w') as f:\n    json.dump(per_class_metrics, f, indent=2)\n\nworst = sorted(per_class_metrics.items(), key=lambda x: x[1]['f1'])[:5]\nprint(f'✓ Top 5 worst F1:')\nfor name, m in worst:\n    print(f'  {name}: {m[\"f1\"]:.4f}')",
            "if len(all_images_wrong) > 0:\n    print('Top erreurs...')\n    sorted_indices = np.argsort(all_wrong_conf)[::-1][:20]\n    \n    fig, axes = plt.subplots(4, 5, figsize=(16, 12))\n    fig.suptitle('Top 20 Erreurs', fontsize=14, fontweight='bold')\n    axes = axes.flatten()\n    \n    for i, idx in enumerate(sorted_indices):\n        ax = axes[i]\n        img = all_images_wrong[idx].numpy().transpose(1, 2, 0)\n        img = (img - img.min()) / (img.max() - img.min())\n        \n        ax.imshow(img)\n        pred = class_names[all_wrong_preds[idx]][:15]\n        true = class_names[all_wrong_truths[idx]][:15]\n        conf = all_wrong_conf[idx]\n        \n        ax.set_title(f'P: {pred}\\nT: {true}\\nC: {conf:.2f}', fontsize=8)\n        ax.axis('off')\n    \n    plt.tight_layout()\n    errors_path = RESULTS_PATH / 'top_errors.png'\n    plt.savefig(errors_path, dpi=150, bbox_inches='tight')\n    plt.close()\n    \n    print(f'✓ Top erreurs: {errors_path}')",
            "with open(RESULTS_PATH / 'confusion_matrix.pkl', 'wb') as f:\n    pickle.dump(cm, f)\n\nevaluation_results = {\n    'test_accuracy': float(accuracy),\n    'total_samples': int(len(test_indices)),\n    'errors': int(len(all_images_wrong)),\n    'per_class_metrics': per_class_metrics\n}\n\nwith open(RESULTS_PATH / 'evaluation_results.json', 'w') as f:\n    json.dump(evaluation_results, f, indent=2)\n\nprint('\\n✓ ÉTAPE 3 COMPLÉTÉE')"
        ]
    ],
    "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}, "language_info": {"name": "python", "version": "3.10.0"}},
    "nbformat": 4,
    "nbformat_minor": 4
}

BASE_PATH = Path('/content/drive/MyDrive/AnanthiX_AI')
NOTEBOOKS_PATH = BASE_PATH / 'notebooks'
NOTEBOOKS_PATH.mkdir(parents=True, exist_ok=True)

notebook_path = NOTEBOOKS_PATH / '03_evaluation.ipynb'
with open(notebook_path, 'w') as f:
    json.dump(notebook_content, f, indent=2)

print(f"✓ Notebook créé : {notebook_path}")

✓ Notebook créé : /content/drive/MyDrive/AnanthiX_AI/notebooks/03_evaluation.ipynb
